<a href="https://colab.research.google.com/github/PozieSwagger/lis4693/blob/main/Lab-5/Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Topic Modeling

## Task 1: Install and Import

Installing the required packages and import the exported lens csv file from the GitHub file.

In [3]:
!pip install gensim
!pip install pyldavis
!pip install nltk

In [15]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import gensim.corpora as corpora
from gensim.models import LdaModel
import re

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
import pandas as pd
import requests
import io

url = "https://raw.githubusercontent.com/PozieSwagger/lis4693/refs/heads/main/Lab-5/lens-export.csv"
response = requests.get(url)
response.raise_for_status()
text = response.text

## Task 2: Display Dataset

Displaying the first 10 rows and cleaning the dataset

Selecting the 'Title' column for topic modeling. Reason for using the 'Title' column is that it usually provides a concise summary of the document's content, making it a good candidate for extracting main topics. While 'Abstract' could also be used, 'Title' often contains key terms that are highly indicative of the document's primary subject. Inaddition, 'Title' column is more likely to have some sort of entry compared to 'Abstract' or 'Keyword'

In [16]:
df = pd.read_csv(io.StringIO(text))
print(df.head(10))

topicModeling = df['Title']
print(topicModeling.head())

               Lens ID                                              Title  \
0  000-012-970-179-311  Quantitative Insights into English Language Le...   
1  000-029-966-419-184  Learning Multilingual Word Representations usi...   
2  000-212-702-600-807  AAAI - Multilingual Transfer Learning for QA u...   
3  000-236-181-506-255  Perspectives and Experiences of Autistic Multi...   
4  000-325-597-644-453                                              Index   
5  000-490-613-378-01X            A Biliteracy Agenda for Genre Research.   
6  000-505-733-929-823  Translanguaging in Teaching and Learning Scien...   
7  000-569-614-192-112  Machine Learning Techniques for Effective Mult...   
8  000-831-636-667-538  Language Socialization in Bilingual and Multil...   
9  000-896-621-920-614  Who’s Teaching Whom? Co-Learning in Multilingu...   

  Date Published  Publication Year        Publication Type  \
0     2024-11-18            2024.0         journal article   
1     2014-01-08            

## Task 3: Preprocessing

First thing being down is creating a function to clean the text being preprocess. With a goal of removing the empty spaces that appear within the bibliographic metadata and punctuations/numbers.

In [13]:
# Initialize lemmatizer and stopwords
stop_words = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Deal with issue of NaN
    if not isinstance(text, str):
        return []

    text = text.lower()

    # Remove punctuation and numbers
    text = re.sub(r'[^a-z]', ' ', text)
    words = text.split()

    # Remove stopwords
    processed_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words and len(word) > 2]
    return processed_words


Apply the preprocessing and viewing bag-of-words to check if needed a custom set of stopwords. Later, see there is no need to at this time.

In [20]:
# Apply preprocessing to the 'Title' column
processed_data = [preprocess_text(doc) for doc in topicModeling]
dictionary = corpora.Dictionary(processed_data)
corpus = [dictionary.doc2bow(doc) for doc in processed_data]

print("Preprocessing complete. Dictionary and Corpus created.")
print(f"Number of unique tokens: {len(dictionary)}")
print(f"Number of documents in corpus: {len(corpus)}")

print("Viewing bag-of-words")
for num in corpus[0]:
    num = num[0]
    print(f"{num}\t{dictionary[num]}")

Preprocessing complete. Dictionary and Corpus created.
Number of unique tokens: 1762
Number of documents in corpus: 1000
Viewing bag-of-words
0	attitude
1	case
2	classroom
3	english
4	esl
5	insight
6	language
7	learner
8	learning
9	multilingual
10	quantitative
11	rural
12	sindh


Running the LDA models for 10 and 20 topics

In [25]:
# Model 1: 10 topics
print("\nRunning LDA model with 10 topics...")
lda_model_10_topics = LdaModel(corpus=corpus, id2word=dictionary, num_topics=10, random_state=100, update_every=1, chunksize=100, passes=10, alpha='auto', per_word_topics=True)

# Model 2: 20 topics
print("Running LDA model with 20 topics...")
lda_model_20_topics = LdaModel(corpus=corpus, id2word=dictionary, num_topics=20, random_state=100, update_every=1, chunksize=100, passes=10, alpha='auto', per_word_topics=True)

# Analyze resulting LDA model for 10
topic_10 = lda_model_10_topics.get_document_topics(corpus)
for topic_id, score in topic_10[0]:
    print(f"Topic {topic_id}: {score}")
for topic in topic_10[0]:
    terms = lda_model_10_topics.get_topic_terms(topic[0], 5)
    print(f"({topic_10[0]}, {topic_10[1]})")
    for num in terms:
        num = num[0]
        print(f"{num}\t{dictionary[num]}")
    print()

# Analyze resulting LDA model for 20
topic_20 = lda_model_20_topics.get_document_topics(corpus)
for topic_id, score in topic_20[0]:
    print(f"Topic {topic_id}: {score}")
for topic in topic_20[0]:
    terms = lda_model_20_topics.get_topic_terms(topic[0], 5)
    print(f"({topic_20[0]}, {topic_20[1]})")
    for num in terms:
        num = num[0]
        print(f"{num}\t{dictionary[num]}")
    print()


Running LDA model with 10 topics...
Running LDA model with 20 topics...
Topic 0: 0.3773297965526581
Topic 3: 0.013380472548305988
Topic 5: 0.16031648218631744
Topic 6: 0.012766974978148937
Topic 7: 0.4093278646469116
([(0, np.float32(0.37732977)), (3, np.float32(0.013380472)), (5, np.float32(0.16031651)), (6, np.float32(0.012766974)), (7, np.float32(0.40932783))], [(1, np.float32(0.63818765)), (3, np.float32(0.019866722)), (5, np.float32(0.06761676)), (6, np.float32(0.018866785)), (7, np.float32(0.21839908))])
7	learner
166	code
938	year
341	awareness
172	switching

([(0, np.float32(0.37733498)), (3, np.float32(0.013380472)), (5, np.float32(0.16029641)), (6, np.float32(0.012766973)), (7, np.float32(0.40934268))], [(1, np.float32(0.63810694)), (3, np.float32(0.019866858)), (5, np.float32(0.067515925)), (6, np.float32(0.018866736)), (7, np.float32(0.21858044))])
80	self
79	motivation
239	content
184	chinese
247	resource

([(0, np.float32(0.37732416)), (3, np.float32(0.013380472)), (5, n

## Task 4: Visualization of Topics